# 01 · Models: research and plan

Oscar has the data ready (`report/02-Data.ipynb`): 27,142 reports, each section turned into FinBERT
chunk vectors, with the shared training and test years. Louis is starting on an MLP. My part is the
modelling around it:

1. **how every model is scored**, so all of us are judged the same way;
2. **the simplest text model**, a linear one, which tells us whether the text helps at all and sets
   the bar the neural networks have to clear;
3. then **the main model**, the attention network.

As in `Oscar/01-Planning.ipynb`, I research first, decide, then build. This notebook is the research
and the plan; the building and results are in **`02-Models.ipynb`**. Section 3.2 updates the plan with
what the first models showed.

## 1. What the brief asks of the models

The brief (Assessment 2) marks four things:

- **(a) implementation**: credit for going beyond off-the-shelf, *exploring multiple implementations*,
  and *examining the mathematical details of choices*;
- **(b) application**: the dataset, how it compares with a real setting, plotting inputs and outputs;
- **(c) correctness** of the methods;
- **(d) robustness** of the results in supporting the conclusions.

And: *"you need to perform robustly in all areas and additionally demonstrate insight somewhere."*

What this means for modelling:

- **Every model is scored the same way**, on several test years, with a measure of uncertainty.
  One test year and no interval is weak on (c) and (d), however good the model.
- **A ladder of models**, each adding one thing, so any gain can be traced to its cause. If a neural
  network beats the baseline, we need to know whether the *network* helped or just the *text*.
- **The maths of each choice** is written down: why this loss, this penalty, this pooling.

## 2. Research

### 2.1 What we predict and how it is scored

We follow Kogan et al. (2009), who built the dataset (`report/01-Introduction.ipynb`):

- **Target:** `logvol_after`, the log of the standard deviation of daily returns over the 12 months
  after the report was filed.
- **Score:** mean squared error (MSE) of the predicted log volatility on a held-out year.
- **Test years** 2001–2006, each one trained on the five years before it.
- **Their baseline** is simply *copying* last year's log volatility. Their only fitted model on last
  year's volatility alone (support vector regression) did worse than copying (Table 2: an MSE of 5.68 in
  2006 with an intercept), so the paper has no working fitted history-only baseline.
- **Their result** (Table 2): models using text plus last year's volatility beat copying in four of the
  six test years (2003–06), for 2006 an MSE of 0.1402 against 0.1463, but **not** in 2001 or 2002, where
  every combined model is worse. (`report/01-Introduction.ipynb` currently says "every test year"; the
  table shows otherwise.)

**Why log volatility and not volatility itself?** Volatility is strongly right-skewed: a few very
volatile companies sit far out in the tail. Squared errors on raw volatility would be dominated by
those few companies, so the score would mostly measure how well we predict outliers. On the log scale
the distribution is close to symmetric, and an error of "twice as volatile" counts the same as "half
as volatile". `02-Models.ipynb` (section 2) checks the skew on our data. Scoring on the log scale also lets us compare
with the paper's numbers directly.

So we score on the actual volatility labels, as a regression (a number, not a category). A
classification framing ("will this company be in next year's riskiest 10%?") is a possible extra
analysis later, scored with ROC AUC, but not the main one: it throws away information and loses the
comparison with the paper.

### 2.2 Which years to train on

`report/02-Data.ipynb` fixed two training windows for each test year $T$:

- **paper:** the five years $T-5 \dots T-1$, as Kogan et al., for direct comparison;
- **strict:** $T-5 \dots T-2$, leaving a one-year gap.

**Why the gap.** A report's label covers the 12 months *after* filing. A report filed in December
2005 has a label running to December 2006, so it overlaps the 2006 test year. In real use, when a
2006 report arrives, that 2005 label is not yet known. Training on it uses information from the
future, and it lets the model learn 2006's market-wide mood from training data.

**What I add: more data, or more recent data?** Markets and reporting language change over time, so
it is not obvious that older reports help. I compare two extra windows, both with the strict gap:

- **last year:** only year $T-2$ (about 2,000–3,000 reports);
- **all prior:** every year from 1996 to $T-2$ (up to about 14,000 reports).

Together with *strict* (four years), this traces a **learning curve**: how error changes as the
training data grows. That links directly to the brief's scaling question. If error is still falling
as we add years, more data would genuinely help at 1000× the scale. If it has flattened, the extra
data would only cost compute.

One caution: a single year is small. A linear model copes, but a neural network may overfit, so
"last year is worse" for a network could mean "small data" rather than "old data is better".

### 2.3 How to tell whether a difference is real

Two models' MSEs on the same test year always differ a little by chance. To say one is better we need
an interval on the **difference**.

For test report $i$, let $d_i = (y_i - \hat y_i^{\text{model}})^2 - (y_i - \hat y_i^{\text{base}})^2$.
The difference in MSE is the mean $\bar d$. A **paired bootstrap** resamples the test reports with
replacement many times and recomputes $\bar d$ each time (Efron & Tibshirani, 1993). The middle 95% of
those values is the interval. "Paired" means both models are scored on the same resampled reports, so
the noise from which reports happen to be in the test year cancels. This is the same idea as the
Diebold–Mariano test for comparing forecasts (Diebold & Mariano, 1995).

**Caveat.** The bootstrap treats reports as independent, but reports in the same year share the
market's shocks: in a crash year everyone's volatility rises together. So each interval is somewhat
too narrow. The stronger evidence is **consistency across the six test years**: a model that wins in
6 of 6 years is convincing even if each single interval is optimistic.

### 2.4 The first text model: which linear model?

**Why linear first.** It is the cheapest test of whether FinBERT's vectors carry *any* information
about future risk. It is also the bar for the neural networks: if a linear model on the same inputs
does as well as an MLP, the network added nothing.

The input per report is 768 numbers per section (1,536 with both sections). These numbers are heavily
correlated with each other, and in the *last year* window there are only about 2,500 training reports.
The candidates:

| Model | Loss + penalty | Fit here? |
|---|---|---|
| Ordinary least squares | squared error, no penalty | **No.** With $p$ = 768–1,536 correlated inputs and a few thousand reports, $X^\top X$ is close to singular and the coefficients are mostly noise |
| **Ridge** | squared error + $\lambda \lVert\beta\rVert_2^2$ | **Yes.** Keeps every input but shrinks them; closed-form solution; one hyperparameter, chosen cheaply |
| Lasso | squared error + $\lambda \lVert\beta\rVert_1$ | Less suited. Sets most coefficients to zero, which assumes a few inputs matter. FinBERT's dimensions have no individual meaning and the information is spread across them |
| Linear support vector regression | $\varepsilon$-insensitive + $\lambda \lVert\beta\rVert_2^2$ | What Kogan et al. used, on word counts. Same penalty as ridge, different loss. Kept for the word-count reproduction |
| Principal components regression | squared error on the top $k$ components | Ridge does the same thing smoothly (see below), without choosing $k$ |

**Decision: ridge.**

**The maths.** Ridge chooses
$$\hat\beta_\lambda = \arg\min_\beta \; \lVert y - X\beta\rVert^2 + \lambda\lVert\beta\rVert^2
= (X^\top X + \lambda I)^{-1} X^\top y.$$
Adding $\lambda I$ makes the matrix invertible however correlated the inputs are. Writing the singular
value decomposition $X = U D V^\top$ shows what the penalty does (Hastie, Tibshirani & Friedman, 2009,
§3.4.1):
$$X\hat\beta_\lambda = \sum_j u_j \, \frac{d_j^2}{d_j^2 + \lambda} \, u_j^\top y .$$
Each direction $u_j$ in the data is kept in proportion $d_j^2/(d_j^2+\lambda)$. Directions with a
lot of spread across reports ($d_j$ large) are kept almost whole, and directions with little spread,
where the fit would mostly chase noise, are shrunk towards zero. Principal components regression keeps
the top $k$ directions whole and drops the rest; ridge is the smooth version of that. Ridge is also the
most likely $\beta$ under a Gaussian prior $\beta \sim N(0, \sigma^2/\lambda \, I)$.

Two practical consequences:

- **Standardise the inputs** (using the training rows only): the penalty treats every coefficient the
  same, so an input measured on a larger scale would be penalised less.
- **Choosing $\lambda$.** Leave-one-out cross-validation is almost free for ridge, because the error
  on report $i$ when it is left out is $e_i / (1 - h_{ii})$, where $h_{ii}$ is the diagonal of the "hat"
  matrix $X(X^\top X + \lambda I)^{-1}X^\top$. One decomposition gives every $\lambda$ at once
  (scikit-learn's `RidgeCV`). `report/02` proposes tuning on the last training year instead, but the
  *last year* window has only one year, so I use leave-one-out in every window for a fair comparison.
  In `02-Models.ipynb` (section 5) I replace `RidgeCV` with generalised cross-validation computed from
  per-year sums. It is a close approximation to leave-one-out, and section 10 there shows it leads to the
  same conclusions.

### 2.5 How a report goes into the linear model

**The text.** A report has a variable number of chunk vectors (median 32). A linear model needs a fixed
length, so I **average each section's chunks** (mean pooling): the `X` table in Oscar's `data.pkl`.
Alternatives are the maximum over chunks, the first chunk only, or a learned weighting. The learned
weighting is our attention network, and mean pooling is exactly that network with all weights equal.
So the comparison "attention vs mean" measures exactly what learning the weights buys.

**Last year's volatility.** Rather than put it in the ridge alongside the 768 text inputs, I fit in two
stages:

1. fit the straight line $\hat y^{\text{line}} = a + b \cdot \text{logvol\_before}$ (the baseline);
2. fit ridge on the text to the **residual** $y - \hat y^{\text{line}}$: the part of next year's
   volatility that last year's does not explain.

Prediction = line + ridge. Three reasons:

- The baseline coefficient is not penalised. In one ridge with standardised inputs, it would be shrunk
  like any text input.
- As $\lambda \to \infty$ the ridge part goes to zero and the model becomes *exactly* the baseline. So
  "ridge vs line" measures what the text adds, and ridge can only move away from the baseline where the
  data supports it.
- The residual is the project's question: what the text says *beyond* last year's volatility.

**Three versions:** Item 1 only, Item 7 only, and both (the two averages side by side, 1,536 inputs).
This gives a first answer to which section carries the risk information.

**A known weakness to watch.** The vectors are FinBERT's CLS vector from `ProsusAI/finbert`, a model
further trained to label financial sentences as positive, negative or neutral (Araci, 2019). Its CLS
vector may mostly encode tone rather than what the text is about. If ridge finds little signal, that
is the first suspect, and the fix is a different encoder: the second FinBERT of Yang et al. (2020),
trained on 10-Ks, earnings calls and analyst reports.

What FETILDA (Xia et al., 2022) found on these two choices, both on a different task (banks' return on
assets from Item 7, not volatility):

- **Which FinBERT** (their Table 6): Yang et al.'s beat Araci's, the one we use, but only slightly (test
  MSE 0.00075 against 0.00078, about 4%), on one target, and with FinBERT fine-tuned rather than frozen as
  ours is. Weak evidence, so the comparison is worth running only if our own results point to the encoder.
- **Which vector per chunk** (their Table 7): the CLS vector beat averaging or taking the maximum of the
  token vectors, which supports the choice in `report/02`.
- On Kogan-style volatility data they also found that fine-tuning FinBERT beat keeping it frozen, by
  about 5% on average.

## 3. Plan

### 3.1 The plan before building

| Step | What | Where | Status |
|---|---|---|---|
| 1 | Shared scoring: windows, MSE, paired bootstrap, baselines | `Alex/scoring.py` | done |
| 2 | Baselines and ridge on averaged FinBERT vectors, Item 1 / Item 7 / both | `02-Models.ipynb` | done |
| 3 | Attention network: learns a weight per chunk, predicts the residual over the line | `02-Models.ipynb` | next |
| 4 | Depending on step 2: tune the attention network, **or**, if the vectors look weak, compare a second encoder on Colab | | after 3 |

**Decision point after step 2.**

- If ridge clearly beats the line, FinBERT's vectors are useful: put the effort into how chunks are
  combined (step 3 and its tweaks).
- If it does not, the vectors are the bottleneck: compare encoders before tuning networks.

**For Louis (and anyone adding a model).** Write the model as `predict(df, train, test)` returning
predictions for the test rows, then:

```python
import pickle, sys; sys.path.insert(0, "../Alex")
import scoring
with open("../data/processed/data.pkl", "rb") as f:
    d = pickle.load(f)
df = scoring.frame(d)
results = scoring.evaluate(df, predict, name="MLP")
```

That scores it on every test year and window, against the same straight-line baseline, with the same
intervals, so our results tables line up.

### 3.2 Updated plan (14 November), after the linear models

**What the first models showed** (`02-Models.ipynb`, section 11):

- Most of the error is the **year's overall volatility level**, which no single report can know. A model
  trained on several years together can use the text to guess the era, which looks like large gains and
  losses that are really about the year. Every model now reports its error split into **level** and
  **spread** (how well it ranks companies within a year).
- Within a year, the averaged FinBERT vectors add **little, and not consistently**: −6% to +10% by year,
  −0.6% to +2.6% on average. By the decision rule in 3.1, the averaged vectors look like the weak point.
- On Kogan et al.'s own full data and windows, a **straight line on last year's volatility, with no
  text**, scores 0.1403 for 2006 against their text model's 0.1402. Much of the paper's gain over its
  baseline comes from fitting history, not from the text. So the fitted line is the honest bar.
- The 11% of reports not matched to EDGAR-CORPUS are the **hard cases**: copying last year gives 3–6 times
  the error there. Our MSEs are therefore lower than the paper's for reasons unrelated to any model.

**Next steps** (content freeze Monday 30 November, 12:00):

| When | What | Who |
|---|---|---|
| Now | Commit and push, so Colab and the others can use the scoring code | Alex |
| Now | To Oscar: the dropped reports are the hard cases (a limitation for `report/02`); "every test year" in `report/01` contradicts Kogan's Table 2; please commit the word-count and attention test code behind `PLAN.md` | Alex |
| Group | Agree: report against both copying and the straight line, with the line as the bar; one rule for tuning hyperparameters (validation year or cross-validation); whether the second FinBERT goes back into the plan | all |
| 16–18 Nov | **Attention network** in `02-Models.ipynb`: each report's chunk vectors (`X_item1`, `X_item7` in `data.pkl`), attention pooling with padding masked (Ilse et al., 2018), predicting the **within-year residual** over the line; dropout, early stopping on the last training year, 5 seeds; Colab GPU; paper and strict windows; Item 1, Item 7, both; plots of the attention weights | Alex |
| 16–18 Nov | MLP, scored with `scoring.evaluate` against the within-year ridge, with the level/spread split | Louis |
| 19–20 Nov | **Decision:** if attention is no better than the within-year ridge, embed Item 7 with Yang et al.'s FinBERT on Colab, timing it on CPU and GPU across batch sizes (this doubles as the scaling experiment) | Alex, Oscar |
| Optional | Embed Kogan's own Item 7 for the ~2,000 dropped reports, so Item 7 models can be scored on the paper's full test sets | Oscar |
| 23–27 Nov | Scaling write-up: map/reduce for FinBERT and for the per-year ridge sums, with the timings | Alex, Oscar |
| by 30 Nov | Models notebook into `report/03-Models.ipynb`; every notebook rerun top to bottom with outputs | all |

## References

- Araci, D. (2019). FinBERT: Financial sentiment analysis with pre-trained language models. *arXiv:1908.10063*.
- Diebold, F. X. & Mariano, R. S. (1995). Comparing predictive accuracy. *Journal of Business & Economic Statistics*, 13(3), 253–263.
- Efron, B. & Tibshirani, R. J. (1993). *An Introduction to the Bootstrap*. Chapman & Hall.
- Golub, G. H., Heath, M. & Wahba, G. (1979). Generalized cross-validation as a method for choosing a good ridge parameter. *Technometrics*, 21(2), 215–223.
- Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning*, 2nd ed., §3.4 (shrinkage methods). Springer.
- Ilse, M., Tomczak, J. M. & Welling, M. (2018). Attention-based deep multiple instance learning. *ICML 2018*.
- Kogan, S., Levin, D., Routledge, B. R., Sagi, J. S. & Smith, N. A. (2009). Predicting risk from financial reports with regression. *NAACL-HLT 2009*, 272–280. https://aclanthology.org/N09-1031/
- Xia, B., Rawte, V. D., Zaki, M. J. & Gupta, A. (2022). FETILDA: An effective framework for fin-tuned embeddings for long financial text documents. *arXiv:2206.06952*.
- Yang, Y., Uy, M. C. S. & Huang, A. (2020). FinBERT: A pretrained language model for financial communications. *arXiv:2006.08097*.